# 01_01 A language model without a brain

*📖 Book: chapter 1*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **the basics** (chapter 1): the ideas every stage uses
>
> Project files this connects to:
>
> - `data/homer/odyssey/greek/book_01.txt`
>
> **This notebook:** What is the simplest program that can guess the next character of Homer? Building it shows what a real model must do better.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to build the simplest language model: by counting
- How it writes Greek, one character at a time
- Why it gets stuck or writes nonsense
- What a better model would need

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **bigram** | a pair of neighbours: one character and the one right after it | `ν` then a space |
| **probability** | how likely something is, from 0 (never) to 1 (always); often called the chance | 0.5 = half the time |
| **greedy** | always picking the single most likely next character | after `ἄ` always `λ` |
| **sampling** | picking the next character at random, but with likely ones picked more often | like drawing from a bag |

## 3. The idea

### ❓ What is the simplest way to guess the next character?

Look at what usually came next in the text itself. Say that in Book 1 the character `ν` is followed by a space 529 times. Then after a `ν`, a space is a pretty good guess.

Counting pairs like this (one character and the one right after it) is called a **bigram** count. *Bi* means two.

### ❓ Is that really a language model?

Yes, it is. It reads text and guesses what comes next. Nothing in it learns anything, though. It's just a table of counts, which is why we call it a language model **without a brain**. It's the baseline the real model has to beat.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Which pairs are in a tiny piece of text?

Let's start with the first two words, `ἄνδρα μοι`. Here's each character with the one right after it:

```
'ἄ'→'ν'  'ν'→'δ'  'δ'→'ρ'  'ρ'→'α'  'α'→' '  ' '→'μ'  'μ'→'ο'  'ο'→'ι'
```

Each pair is one small fact, like "after `ν` came `δ`, once". And yes, the space counts as a character too.

👉 **Predict first:** 9 characters. How many pairs?

In [ ]:
from itertools import pairwise

for current, nxt in pairwise("ἄνδρα μοι"):
    print(f"{current!r}→{nxt!r}", end="  ")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

So, 8 pairs from 9 characters, because the last character has nothing after it. `pairwise` hands them over one at a time. Next we do the same for all of Book 1, and count.

</details>

### ❓ How do I count which character follows which?

Walk along the text in pairs, each character with the one after it. For every pair we add 1 to a counter, and what we end up with is a table of counts.

👉 **Predict first:** after `ν`, what do you think comes most often?

In [ ]:
from collections import Counter, defaultdict
from itertools import pairwise

from homer_ai.train_homer import BOOKS_DIR, load_books

text = load_books(BOOKS_DIR)[0]  # Book 1 of the Odyssey

follows = defaultdict(Counter)  # for each character: a counter of what came next
for current, nxt in pairwise(text):
    follows[current][nxt] += 1

print(follows["ν"].most_common(5))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

There it is: `(' ', 529)` comes first. That's `ν` followed by a space, because so many Greek words end in **-ν**.

</details>

### ❓ Can I see those counts?

Sure. A picture shows how lopsided they are. These are the 8 most common characters after `ν`.

👉 **Predict first:** will the space bar be a little taller than the rest, or much taller?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

top = follows["ν"].most_common(8)
total = sum(follows["ν"].values())

fig = studio_figure(
    "What follows one character, in Book 1",
    f"After ν, a space comes {top[0][1]} times out of {total:,}",
)
ax = studio_axes(fig)
ax.set_axisbelow(True)  # grid lines behind the bars
bars = ax.bar([repr(c) for c, _ in top], [n for _, n in top], color=MUTED)
bars[0].set_color(ACCENT)
for bar, (_, n) in zip(bars, top, strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2, n + 6, str(n),
        ha="center", color=INK, family="monospace", fontsize=11,
    )
ax.tick_params(axis="x", labelsize=14, labelcolor=INK)
ax.grid(axis="x", visible=False)
ax.set_ylabel("times it came next")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The space bar is far taller than the others, so a space is the best guess. But look at the rest. Plenty of other characters follow `ν` too, and the next steps show what that does to us.

</details>

### ❓ What if it always picks the most likely next character?

That's called **greedy**. We start with `ἄ` (the Odyssey's first character) and do it 40 times.

👉 **Predict first:** after `ἄ`, the most common next character is `λ`. What happens next?

In [ ]:
line = "ἄ"
for _ in range(40):
    last = line[-1]
    best = follows[last].most_common(1)  # e.g. [('λ', 20)]: one (character, count) pair
    line += best[0][0]  # [0] = that pair, then [0] again = its character

print(line)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`ἄλλλλλλλ…`. After `λ`, the most common next character is `λ` again (think of `πολλὰ`), so greedy picks it forever. It's stuck in a loop. Our real model loops too, by the way: you'll see `ἴδεν ἴδεν ἴδεν` and `και και και` later on.

</details>

### ❓ Your turn: which character most often follows `κ`?

The greedy loop asked `follows` for the most common next character, over and over. Now you ask it once, for `κ`. We want the character alone, not the pair with its count.

👉 **Predict first:** think of a short Greek word that starts with `κ` and turns up everywhere.

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: the one character that most often comes after "κ" in Book 1
# 💡 hint: most_common(1) gives a list with one (character, count) pair
after_kappa = None

In [ ]:
from homer_ai.lessons import check

check(
    "the character after κ",
    after_kappa,
    max(follows["κ"], key=follows["κ"].get),
    coach="most_common(1) gives a list like [('α', 140)]. Take [0] for the pair, "
    "then [0] again for its character.",
)

<details><summary>💡 Hint</summary>

Look at the two lines inside the greedy loop: `best = follows[last].most_common(1)` and then `best[0][0]`. Here `last` is `"κ"`.

</details>

<details><summary>✅ Answer</summary>

```python
after_kappa = follows["κ"].most_common(1)[0][0]
```

It's `α`, 140 times out of 432. That's mostly `καὶ` (and), one of the most common words in the book.

</details>

### ❓ How can it pick differently each time?

With **sampling**: pick at random, but in proportion to the counts. If `λ` came after `ἄ` 20 times and `γ` 10 times, then `λ` gets picked twice as often as `γ`, but `γ` still gets its chance. `random.choices` does exactly this.

`random.seed(42)` makes the "random" picks the same on every run, so your output matches this page.

👉 **Predict first:** will it write real Greek words?

In [ ]:
import random

random.seed(42)
line = "ἄ"
for _ in range(60):
    options = follows[line[-1]]
    line += random.choices(list(options), weights=list(options.values()))[0]

print(line)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`ἄγχαὶ τʼ ἀν καὶ ν γῆιδαίρτεροσθοἴ ἕσιχοσισαὶ ἔπε ἥρ ἐπὶ θεσε`. It does look Greek. Real little words turn up (`καὶ`, `ἐπὶ`) and the accents land in sensible places. Most of it is nonsense, though, because it only ever looks **one** character back.

</details>

## 5. Full example

### ❓ How sure is it about each guess?

Turn the counts after one character into **probabilities**: divide each count by the total, and they always add up to 1. (In the code, `{character!r}` prints the character in quotes, so a space shows up as `' '`.)

👉 **Predict first:** after a space, how likely is the most common next character?

In [ ]:
after_space = follows[" "]
total = sum(after_space.values())

for character, count in after_space.most_common(5):
    chance = count / total
    print(f"{character!r}: {count} times, probability {chance:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

So after a space, `δ` is the most common next character, but its probability is only about 0.09. That's fewer than 1 time in 10. Lots of characters can start a word, so one character of context just isn't enough to be sure.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ How could it do better?

By looking further back. After `ἄνδρα μο` the next character is almost certainly `ι`. After just `ο` it could be lots of things. The trouble is that counting every possible 8-character history would need an enormous table, and most histories never show up in Book 1 anyway. So we build a model that **learns** patterns and can look many characters back. That's the rest of the course.

## 6. Recap

- Counting which character follows which (a bigram count) is already a language model.
- Greedy picking gets stuck in loops; sampling looks Greek but is mostly nonsense.
- It only looks one character back: a real model must look further, and learn.

**You can now:**

- count which character follows which, with `pairwise` and a `Counter`
- write text from the counts, greedy or by sampling
- turn counts into probabilities that add up to 1

**Next:** [01_02: Neural networks](01_02_neural_networks.ipynb).